In [ ]:
import anywidget
import traitlets
import os
import base64
import mimetypes

class FileDownloader(anywidget.AnyWidget):
    """
    An anywidget that renders a button. When clicked, it triggers a server-side
    read of 'file_path' and sends the content to the browser for download.

    The button starts disabled and only enables after a successful handshake
    with the Python kernel, ensuring it doesn't appear active in a dead notebook.
    """

    # The path to the file on the server/local disk that you want to download
    file_path = traitlets.Unicode(help="Path to the file to be downloaded").tag(sync=True)

    # Label for the button
    button_text = traitlets.Unicode("Download File").tag(sync=True)

    _esm = """
    export function render({ model, el }) {
      // Create the button element
      let btn = document.createElement("button");
      btn.classList.add("jupyter-widgets", "jupyter-button", "widget-button");
      btn.style.width = "100%";

      // Initial state: Disabled and waiting
      btn.innerText = "Waiting for Kernel...";
      btn.disabled = true;

      // Update button text if the Python trait changes
      model.on("change:button_text", () => {
        // Only update visually if we are already connected/enabled
        if (!btn.disabled) {
            btn.innerText = model.get("button_text");
        }
      });

      // Handle the click event
      btn.addEventListener("click", () => {
        const filePath = model.get("file_path");

        if (!filePath) {
            alert("No file path set in the Python widget!");
            return;
        }

        // Disable button and show loading state
        const originalText = btn.innerText;
        btn.innerText = "Downloading...";
        btn.disabled = true;

        // Send a request message to the Python backend
        model.send({ type: "request_download" });

        // Helper to restore button state
        const restoreBtn = () => {
            btn.innerText = originalText;
            btn.disabled = false;
        };

        // Timeout safety to restore button if Python doesn't respond within 5s
        setTimeout(restoreBtn, 5000);
      });

      el.appendChild(btn);

      // Listen for messages coming from Python
      model.on("msg:custom", (msg) => {
        if (msg.type === "connection_verified") {
            // HANDSHAKE COMPLETE: Kernel is alive.
            btn.disabled = false;
            btn.innerText = model.get("button_text");
        }
        else if (msg.type === "file_content") {
            // 1. Create a Blob from the Base64 data
            const byteCharacters = atob(msg.content);
            const byteNumbers = new Array(byteCharacters.length);
            for (let i = 0; i < byteCharacters.length; i++) {
                byteNumbers[i] = byteCharacters.charCodeAt(i);
            }
            const byteArray = new Uint8Array(byteNumbers);
            const blob = new Blob([byteArray], { type: msg.mime_type });

            // 2. Create a temporary link to trigger the download
            const url = window.URL.createObjectURL(blob);
            const a = document.createElement("a");
            a.style.display = "none";
            a.href = url;
            a.download = msg.filename;
            document.body.appendChild(a);
            a.click();

            // 3. Cleanup
            window.URL.revokeObjectURL(url);
            document.body.removeChild(a);

            // Restore button text
            btn.innerText = model.get("button_text");
            btn.disabled = false;

        } else if (msg.type === "error") {
            alert(`Error: ${msg.message}`);
            btn.innerText = model.get("button_text");
            btn.disabled = false;
        }
      });

      // INITIATE HANDSHAKE
      // Send a message to Python to check if the kernel is listening.
      // If the kernel is dead (saved notebook), this message goes nowhere,
      // and the button remains disabled.
      setTimeout(() => {
        model.send({ type: "check_connection" });
      }, 500);
    }
    """

    def __init__(self, file_path=None, **kwargs):
        super().__init__(**kwargs)
        if file_path:
            self.file_path = file_path

        # Register the message handler
        self.on_msg(self._handle_custom_msg)

    def _handle_custom_msg(self, msg, content):
        """
        Callback for when the frontend sends a message to Python.
        """
        msg_type = msg.get("type")

        if msg_type == "check_connection":
            # Reply to the frontend to confirm we are alive
            self.send({"type": "connection_verified"})

        elif msg_type == "request_download":
            self._process_download()

    def _process_download(self):
        """
        Reads the file from disk and sends it to the frontend.
        """
        target_path = self.file_path

        # Basic validation
        if not target_path:
            self.send({"type": "error", "message": "File path is not defined."})
            return

        if not os.path.exists(target_path):
            self.send({"type": "error", "message": f"File not found: {target_path}"})
            return

        try:
            # Guess the MIME type so the browser handles it correctly
            mime_type, _ = mimetypes.guess_type(target_path)
            if mime_type is None:
                mime_type = 'application/octet-stream'

            # Read and encode the file
            with open(target_path, "rb") as f:
                file_content = f.read()

            b64_content = base64.b64encode(file_content).decode("utf-8")

            # Send back to JS
            self.send({
                "type": "file_content",
                "filename": os.path.basename(target_path),
                "mime_type": mime_type,
                "content": b64_content
            })

        except Exception as e:
            self.send({"type": "error", "message": str(e)})

dummy_filename = "example_data.txt"
with open(dummy_filename, "w") as f:
    f.write("Hello! This is a file dynamically read from the kernel disk.\n")
    f.write("If you are reading this, the widget worked.")

FileDownloader(file_path=dummy_filename)

FileDownloader(file_path='example_data.txt')